# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [4]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: /workspaces/llm-data-analysis-study/llm-data-analysis-course/notebooks
프로젝트 루트: /workspaces/llm-data-analysis-study/llm-data-analysis-course
전처리 데이터 폴더: /workspaces/llm-data-analysis-study/llm-data-analysis-course/data/processed
보고서 폴더: /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [5]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [6]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (96, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (752, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [7]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [8]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [9]:
customers['age'].describe()


count    150.000000
mean      43.913333
std       17.802397
min       18.000000
25%       30.000000
50%       41.000000
75%       56.750000
max      150.000000
Name: age, dtype: float64

In [10]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [11]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [12]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,패션,16,16.67
1,뷰티,16,16.67
2,식품,16,16.67
3,도서,13,13.54
4,전자기기,13,13.54
5,스포츠,10,10.42
6,생활용품,9,9.38
7,전자 기기,1,1.04
8,생활 용품,1,1.04
9,뷰 티,1,1.04


In [13]:
products['price'].describe()


count    9.600000e+01
mean     2.048958e+05
std      1.011848e+06
min      5.000000e+03
25%      5.275000e+04
50%      9.450000e+04
75%      1.532500e+05
max      9.999999e+06
Name: price, dtype: float64

In [14]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
2,뷰티,16,710937.0,18000.0,9999999.0
3,생활 용품,1,124000.0,124000.0,124000.0
6,식품,16,120812.0,11000.0,195000.0
8,전자기기,13,120231.0,38000.0,178000.0
9,패션,16,115125.0,28000.0,197000.0
0,도서,13,103692.0,34000.0,182000.0
5,스포츠,10,78400.0,5000.0,183000.0
4,생활용품,9,69556.0,29000.0,147000.0
1,뷰 티,1,54000.0,54000.0,54000.0
7,전자 기기,1,21000.0,21000.0,21000.0


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [15]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


In [16]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [17]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00
주문 월 개수: 16


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [18]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [19]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (752, 6)
병합 후 sales_items: (752, 9)
상품명 누락: 26
카테고리 누락: 26


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,39,2.0,172000.0,344000.0,뷰티 상품 039,뷰티,172000.0
1,2,2,26,4.0,79000.0,316000.0,뷰티 상품 026,뷰티,79000.0
2,3,2,46,3.0,87000.0,261000.0,생활용품 상품 046,생활용품,87000.0
3,4,2,80,1.0,187000.0,187000.0,패션 상품 080,패션,187000.0
4,5,2,91,5.0,125000.0,625000.0,전자기기 상품 091,전자기기,125000.0


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [20]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
9,패션,545.0,64890000.0,26.39
6,식품,394.0,47044000.0,19.13
8,전자기기,303.0,37542000.0,15.27
2,뷰티,282.0,28614000.0,11.64
0,도서,277.0,28246000.0,11.49
5,스포츠,254.0,20651000.0,8.40
4,생활용품,213.0,14303000.0,5.82
3,생활 용품,27.0,3348000.0,1.36
1,뷰 티,19.0,1026000.0,0.42
7,전자 기기,12.0,252000.0,0.10


In [21]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,패션,545.0,64890000.0,26.39,115125.0
1,식품,394.0,47044000.0,19.13,120812.0
2,전자기기,303.0,37542000.0,15.27,120231.0
3,뷰티,282.0,28614000.0,11.64,710937.0
4,도서,277.0,28246000.0,11.49,103692.0
5,스포츠,254.0,20651000.0,8.40,78400.0
6,생활용품,213.0,14303000.0,5.82,69556.0
7,생활 용품,27.0,3348000.0,1.36,124000.0
8,뷰 티,19.0,1026000.0,0.42,54000.0
9,전자 기기,12.0,252000.0,0.10,21000.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [22]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales: (752, 12)
order_date 누락: 11


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,39,2.0,172000.0,344000.0,144.0,2026-07-12,naver_pay,completed,2026-07,Sunday
1,2,2,26,4.0,79000.0,316000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
2,3,2,46,3.0,87000.0,261000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
3,4,2,80,1.0,187000.0,187000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
4,5,2,91,5.0,125000.0,625000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday


In [23]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-07,13729000.0,19,722579.0
1,2025-08,32639000.0,25,1305560.0
2,2025-09,13058000.0,12,1088167.0
3,2025-10,16680000.0,22,758182.0
4,2025-11,15080000.0,23,655652.0
5,2025-12,16152000.0,23,702261.0
6,2026-01,19053000.0,26,732808.0
7,2026-02,14207000.0,19,747737.0
8,2026-03,13841000.0,22,629136.0
9,2026-04,21007000.0,26,807962.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [24]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)


,customer_id,name,city,order_count,total_sales,avg_order_value
15,18.0,서하현,대전,2,16666000.0,8333000.0
2,3.0,이예우,서울,7,8909000.0,1272714.0
17,20.0,김성희,성남,6,6200000.0,1033333.0
45,52.0,박유현,울산,5,5388000.0,1077600.0
11,14.0,임서영,대구,4,5204000.0,1301000.0
19,22.0,강영민,부산,4,5029000.0,1257250.0
48,55.0,장유미,부산,6,4850000.0,808333.0
74,90.0,윤서미,수원,5,4847000.0,969400.0
12,15.0,강수영,광주,5,4281000.0,856200.0
77,93.0,윤도우,울산,4,4001000.0,1000250.0


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [25]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [26]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[PosixPath('/workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch06_category_sales.csv'),
 PosixPath('/workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch06_monthly_sales.csv'),
 PosixPath('/workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch06_customer_sales.csv'),
 PosixPath('/workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch06_eda_questions.csv'),
 PosixPath('/workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch06_product_category.csv'),
 PosixPath('/workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch06_customer_city.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [27]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [28]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [29]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
9,패션,545.0,64890000.0,26.39
6,식품,394.0,47044000.0,19.13
8,전자기기,303.0,37542000.0,15.27
2,뷰티,282.0,28614000.0,11.64
0,도서,277.0,28246000.0,11.49


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [ ]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.

# 구매 금액은 완료(completed) 주문의 상세 금액 합계로 계산합니다.
completed_orders = orders.loc[orders['order_status'].eq('completed')].copy()
completed_items = order_items.merge(
    completed_orders[['order_id', 'customer_id']], on='order_id', how='inner'
)
completed_items = completed_items.merge(
    customers[['customer_id', 'city']], on='customer_id', how='left'
)
completed_items['city'] = completed_items['city'].fillna('도시 정보 없음')

city_customer_counts = customers.copy()
city_customer_counts['city'] = city_customer_counts['city'].fillna('도시 정보 없음')
city_customer_counts = city_customer_counts.groupby('city').agg(
    customer_count=('customer_id', 'nunique')
).reset_index()
city_purchase_totals = completed_items.groupby('city', as_index=False).agg(
    total_purchase_amount=('line_total', 'sum')
)
city_comparison = city_customer_counts.merge(city_purchase_totals, on='city', how='outer')
city_comparison['customer_count'] = city_comparison['customer_count'].fillna(0).astype(int)
city_comparison['total_purchase_amount'] = city_comparison['total_purchase_amount'].fillna(0)
city_comparison = city_comparison.sort_values('total_purchase_amount', ascending=False)
city_comparison

In [ ]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.

# 상품 수는 products의 카테고리별 상품 수, 매출은 완료 주문 상세 기준입니다. 카테고리명 원형을 유지하므로 내부 공백 차이(예: 뷰티/뷰 티)는 별도 범주로 표시됩니다.
completed_orders = orders.loc[orders['order_status'].eq('completed'), ['order_id']]
completed_items = order_items.merge(completed_orders, on='order_id', how='inner')
completed_sales_by_product = completed_items.groupby('product_id', as_index=False).agg(
    total_quantity=('quantity', 'sum'), total_sales=('line_total', 'sum')
)
product_category_check = products.merge(
    completed_sales_by_product, on='product_id', how='left'
)
product_category_check['total_quantity'] = product_category_check['total_quantity'].fillna(0)
product_category_check['total_sales'] = product_category_check['total_sales'].fillna(0)
product_category_summary = product_category_check.groupby('category', dropna=False, as_index=False).agg(
    product_count=('product_id', 'nunique'),
    total_quantity=('total_quantity', 'sum'),
    total_sales=('total_sales', 'sum'),
).sort_values('total_sales', ascending=False)

# 상품 ID와 연결되지 않은 완료 주문 상세 매출은 별도 행으로 표시합니다.
known_product_ids = products['product_id'].dropna().unique()
unmatched_items = completed_items.loc[~completed_items['product_id'].isin(known_product_ids)]
if not unmatched_items.empty:
    product_category_summary = pd.concat([
        product_category_summary,
        pd.DataFrame([{
            'category': '상품 정보 없음',
            'product_count': 0,
            'total_quantity': unmatched_items['quantity'].sum(),
            'total_sales': unmatched_items['line_total'].sum(),
        }]),
    ], ignore_index=True)

product_category_summary = product_category_summary.sort_values('total_sales', ascending=False)
product_category_summary

In [ ]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.

# 주문별 상세 금액을 먼저 합산한 뒤, 결제수단별 주문 평균을 계산합니다.
completed_orders = orders.loc[orders['order_status'].eq('completed'), ['order_id', 'payment_method']]
completed_order_amounts = order_items.groupby('order_id', as_index=False).agg(
    order_amount=('line_total', 'sum')
)
payment_order_amounts = completed_orders.merge(
    completed_order_amounts, on='order_id', how='inner'
)
payment_aov = payment_order_amounts.groupby('payment_method', dropna=False, as_index=False).agg(
    order_count=('order_id', 'nunique'),
    total_sales=('order_amount', 'sum'),
    avg_order_value=('order_amount', 'mean'),
)
payment_aov['avg_order_value'] = payment_aov['avg_order_value'].round(0)
payment_aov = payment_aov.sort_values('avg_order_value', ascending=False)
payment_aov

In [ ]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.

# 완료 주문의 상세 행을 주문·상품 정보와 연결해 월별·카테고리별로 집계합니다. 카테고리명 원형을 유지하므로 내부 공백 차이는 별도 범주로 표시됩니다.
completed_orders = orders.loc[orders['order_status'].eq('completed'), ['order_id', 'order_date']].copy()
completed_orders['order_date'] = pd.to_datetime(completed_orders['order_date'], errors='coerce')
completed_orders['order_month'] = completed_orders['order_date'].dt.to_period('M').astype('string')
completed_orders['order_month'] = completed_orders['order_month'].fillna('날짜 미상')
completed_items = order_items.merge(completed_orders, on='order_id', how='inner')
completed_items = completed_items.merge(
    products[['product_id', 'category']], on='product_id', how='left'
)
completed_items['category'] = completed_items['category'].fillna('상품 정보 없음')
monthly_category_sales = completed_items.pivot_table(
    index='order_month', columns='category', values='line_total',
    aggfunc='sum', fill_value=0, dropna=False
).reset_index()
monthly_category_sales

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.
